In [1]:
##### Converts raw raster on ag GDP into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # % of GDP which is ag

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from pathlib import Path
import rasterstats

In [3]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# country geographies 
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# ag GDP raster
ag_GDP = f"{cd}/Data/Raw/Predictors/AgGDP_Ru/aggdp2010.tif"

# GDP raster
GDP = f"{cd}/Data/Raw/Predictors/GDP_Kummu/rast_gdpTot_1990_2022_5arcmin.tif" # band 21 for 2010

# save paths
pixels_pct_ag = f"{cd}/Data/Clean/Predictors/Rasters/pct_GDP_ag.tif"
vectors = f"{cd}/Data/Clean/Predictors/Vectors/pct_GDP_ag.csv"

In [4]:
### Prep using target raster

### PREP 
# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()
    ref_nodata    = ref.nodata
    ref_data      = ref.read(1)

# build reference valid mask: True where reference raster HAS data
if ref_nodata is not None:
    ref_valid = ~np.isnan(ref_data) if np.isnan(ref_nodata) else (ref_data != ref_nodata)
else:
    ref_valid = ~np.isnan(ref_data)


In [5]:
#### Run resampling for pixel scale 

### RESAMPLE 
def resample_to_ref(src_path, band):
    dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, band),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.sum,
            dst_nodata=np.nan,
        )
    dst_array[dst_array == -9999] = np.nan
    return dst_array

ag    = resample_to_ref(ag_GDP, 1)
total = resample_to_ref(GDP, 21)


### CALCULATE 
# compute ag % GDP
with np.errstate(invalid="ignore", divide="ignore"):
    pct_ag = np.where(
        (total > 0) & ~np.isnan(total) & ~np.isnan(ag),
        ag / total,
        np.nan
    )

pct_ag = np.clip(pct_ag, 0, 1).astype(np.float32)


### ALIGN TO REFERENCE MASK

# Step 1: zero out pixels where reference has no data
pct_ag[~ref_valid] = np.nan

# Step 2: identify pixels that need filling
#   — reference has data, but predictor is still NaN
needs_fill = ref_valid & np.isnan(pct_ag)

print(f"Pixels needing fill: {needs_fill.sum()}")

if needs_fill.any():

    # reproject countries to match reference raster CRS
    if countries.crs != dst_crs:
        countries = countries.reindex(countries.index)  
        countries = countries.to_crs(dst_crs)

    # compute country-level mean of pct_ag (ignoring NaNs)
    # use rasterstats with the current (partial) predictor as an in-memory array
    country_stats = rasterstats.zonal_stats(
        countries,
        pct_ag,
        affine=dst_transform,
        stats=["mean"],
        nodata=np.nan,
    )

    # build a lookup: country index → mean pct_ag
    country_means = {
        i: s["mean"] for i, s in enumerate(country_stats)
        if s["mean"] is not None
    }

    # rasterise each country and fill the gap pixels with that country's mean
    fill_array = np.full(dst_shape, np.nan, dtype=np.float32)

    for idx, row in countries.iterrows():
        mean_val = country_means.get(idx)
        if mean_val is None:
            continue  # no data for this country at all; leave as NaN

        # burn this country's geometry into a boolean mask
        geom = [row.geometry]
        country_mask = geometry_mask(
            geom,
            transform=dst_transform,
            invert=True,          # True inside the country polygon
            out_shape=dst_shape,
        )

        # fill only the pixels that need filling AND fall inside this country
        to_fill = needs_fill & country_mask
        fill_array[to_fill] = mean_val

    # apply the fill
    pct_ag = np.where(needs_fill, fill_array, pct_ag)

    # report any pixels still unfilled (country had no valid data at all)
    still_missing = ref_valid & np.isnan(pct_ag)
    if still_missing.any():
        print(f"Warning: {still_missing.sum()} pixels still unfilled after country-average imputation "
              f"(country had no valid pct_ag data). These will be saved as NoData.")


### SAVE
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = pct_ag.copy()
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels_pct_ag, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

Pixels needing fill: 775032
Saved to /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/pct_GDP_ag.tif


In [6]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
gdf_proj = total_geo.to_crs("EPSG:4326")

result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE

# sum irrigation and cropland in each vector
zonal_ag  = rasterstats.zonal_stats(gdf_proj, ag_GDP, stats="sum", nodata=-9999)
zonal_total = rasterstats.zonal_stats(gdf_proj, GDP,   stats="sum", nodata=-9999)

### compute share at vector level
ag_sums  = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_ag])
total_sums = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_total])

with np.errstate(invalid="ignore", divide="ignore"):
    pct = np.where(
        (total_sums > 0) & ~np.isnan(total_sums) & ~np.isnan(ag_sums),
        ag_sums / total_sums,
        np.nan
    )

result["pct_GDP_ag"] = np.clip(pct, 0, 1)

### save
result.to_csv(vectors, index=False)